# Polymarket Strategy Audit

Audit notebook for the current Polymarket crypto-wallet cohort.

Focus wallets:
- `0x8dxd`
- `justdance`
- `BoneReader`
- `coinman2`
- `0xf705fa045201391d9632b7f3cde06a5e24453ca7`

This notebook uses the reusable audit generator in `prediction_market_agents/polymarket_strategy_audit.py` so the same logic can run in the terminal and in an interactive notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "prediction_market_agents").exists() and (candidate / "copy_trader_intel").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from prediction_market_agents.polymarket_strategy_audit import generate_audit, save_audit

audit = generate_audit(root_dir=ROOT)
save_audit(audit)
audit["generated_at"], audit["focus_missing_keys"]

In [ ]:
focus_df = pd.DataFrame(audit["focus_audit"])
focus_df[[
    "focus_key",
    "source",
    "user_name",
    "wallet_archetype",
    "copyability_gate_reason",
    "audit_verdict",
    "latency_arb_score",
    "crypto_profile_score",
    "crypto_recent_score_30d",
    "crypto_top_symbol",
    "crypto_top_symbol_pnl_share",
    "live_pick_count",
    "live_signal_count",
    "risk_flags",
]]

In [ ]:
pd.DataFrame(audit["cohort_summary"]["top_copyable_profiles"])

In [ ]:
archetype_df = pd.DataFrame(
    audit["cohort_summary"]["archetype_counts"].items(),
    columns=["wallet_archetype", "count"],
).sort_values("count", ascending=False)
gate_df = pd.DataFrame(
    audit["cohort_summary"]["gate_reason_counts"].items(),
    columns=["copyability_gate_reason", "count"],
).sort_values("count", ascending=False)
archetype_df, gate_df

## Reading the Audit

- `audit_verdict = copyable_live`: copyable wallet with a live vetted PM pick in the current snapshot.
- `audit_verdict = blocked`: the wallet remains useful for research, but should not be copied under current gating.
- `risk_flags` highlight why a wallet is still dangerous even when the raw PnL or raw win rate looks strong.

Use this notebook to decide whether a wallet belongs in the live copy path, watchlist-only bucket, or the explicit do-not-copy bucket.